# 03 — Model Comparison

**Goal:** Compare classical machine learning models for predicting aqueous solubility (LogS), using the 17 molecular descriptors and 1,024 Morgan fingerprint bits.

Here we compare models using cross-validation on the **training data only**. We will tune models in Notebook 04 and use the held-out test sets only when the final models are ready. This avoids choosing models based on test performance.

## 1. Imports and data

We load the cleaned dataset made in Notebook 01. Notebook 02 already explored the data and tested some initial models, so we do not repeat that work here.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

SEED = 42
RDLogger.DisableLog("rdApp.warning")  # hide repeated RDKit hydrogen warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = "aqsoldb_cleaned_with_morgan.csv"
data_dirs = (PROJECT_ROOT / "data" / "processed",
             PROJECT_ROOT / "data" / "processed_morgan", PROJECT_ROOT)
DATA_PATH = next((folder / DATA_FILE for folder in data_dirs
                  if (folder / DATA_FILE).is_file()), None)
assert DATA_PATH is not None, f"Could not find {DATA_FILE}. Run Notebook 01 or copy the CSV into the project."

df = pd.read_csv(DATA_PATH)
FEATURES = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors",
    "NumHDonors", "NumHeteroatoms", "NumRotatableBonds",
    "NumValenceElectrons", "NumAromaticRings", "NumSaturatedRings",
    "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT"
]
FEATURES += [f"Morgan_{bit:04d}" for bit in range(1024)]
TARGET = "Solubility"
print("Original data:", df.shape)
print("Input features:", len(FEATURES))

## 2. Prepare the data and create fair splits

We use two holdout settings:

- **Connectivity-group split:** Keeps molecules with the same InChIKey connectivity together. This is our random-style comparison.
- **Scaffold split:** Keeps related ring frameworks together, testing generalization to unfamiliar chemical structures.

Notebook 01 excluded two unparseable SMILES when preparing the Morgan CSV. This step checks the remaining structures and saves any additional exclusions. We also make sure that scaffold and connectivity groups cannot overlap across the scaffold holdout.

In [ ]:
# Extract the main ring scaffold for each molecule
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

scaffolds = df["SMILES"].apply(get_scaffold)
excluded = df.loc[scaffolds.isna()].copy()
df_model = df.loc[scaffolds.notna()].copy().reset_index(drop=True)
df_model["scaffold"] = scaffolds.loc[scaffolds.notna()].to_numpy()

X = df_model[FEATURES].copy()
y = df_model[TARGET].copy()

# First 14 InChIKey characters describe molecular connectivity
connectivity = df_model["InChIKey"].str[:14].reset_index(drop=True)

# Molecules without rings get a group based on connectivity
scaffold = df_model["scaffold"].copy()
no_ring = scaffold.eq("")
scaffold.loc[no_ring] = "acyclic_" + connectivity.loc[no_ring]

print("Modeling rows:", len(df_model))
print("Excluded unparseable structures:", len(excluded))
print("Acyclic molecules:", int(no_ring.sum()))
print("Unique scaffold groups:", scaffold.nunique())

### Keep scaffold and connectivity groups together

Sometimes structures with the same connectivity have slightly different scaffold labels. The small grouping step below links molecules that share **either** relationship, so neither can cross the scaffold split.

In [ ]:
# Combine any rows connected by a scaffold OR an InChIKey connectivity group
parent = list(range(len(df_model)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def join(i, j):
    parent[find(i)] = find(j)

for labels in (scaffold, connectivity):
    first_seen = {}
    for i, label in enumerate(labels):
        if label in first_seen:
            join(i, first_seen[label])
        else:
            first_seen[label] = i

combined_groups = pd.Series([find(i) for i in range(len(df_model))])

# Make the two 80/20 group-aware train/test splits
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
random_train_idx, random_test_idx = next(splitter.split(X, y, groups=connectivity))
scaffold_train_idx, scaffold_test_idx = next(splitter.split(X, y, groups=combined_groups))

def group_overlap(labels, train_idx, test_idx):
    return len(set(labels.iloc[train_idx]) & set(labels.iloc[test_idx]))

assert group_overlap(connectivity, random_train_idx, random_test_idx) == 0
assert group_overlap(scaffold, scaffold_train_idx, scaffold_test_idx) == 0
assert group_overlap(connectivity, scaffold_train_idx, scaffold_test_idx) == 0

print("Connectivity split: train", len(random_train_idx), "/ test", len(random_test_idx))
print("Scaffold split:     train", len(scaffold_train_idx), "/ test", len(scaffold_test_idx))
print("Overlap checks passed")

## 3. Models and evaluation metrics

We compare the same **11 models** used in our earlier work. Some models need standardized inputs, so we keep scaling inside their pipelines. This prevents training/validation leakage.

We use **RMSE** as the main metric (lower is better), along with MAE and R². These models have initial settings; we will tune the strongest ones in Notebook 04.

In [ ]:
scaled = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
trees = [("imputer", SimpleImputer(strategy="median"))]

models = {
    "Mean Baseline": Pipeline(trees + [("model", DummyRegressor(strategy="mean"))]),
    "Linear Regression": Pipeline(scaled + [("model", LinearRegression())]),
    "Ridge": Pipeline(scaled + [("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline(scaled + [("model", Lasso(alpha=0.01, max_iter=20000))]),
    "ElasticNet": Pipeline(scaled + [("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000))]),
    "Decision Tree": Pipeline(trees + [("model", DecisionTreeRegressor(max_depth=10, random_state=SEED))]),
    "Random Forest": Pipeline(trees + [("model", RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1))]),
    "Extra Trees": Pipeline(trees + [("model", ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=1))]),
    "Gradient Boosting": Pipeline(trees + [("model", HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=SEED))]),
    "SVR": Pipeline(scaled + [("model", SVR(kernel="rbf", C=10, epsilon=0.1))]),
    "kNN": Pipeline(scaled + [("model", KNeighborsRegressor(n_neighbors=5))])
}

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}
print("Models to compare:", len(models))

## 4. Five-fold cross-validation

We run all models on the **training portion** of each holdout. Each fold keeps relevant molecular groups intact. Test data is saved for final evaluation later.

The two settings have different training subsets, so their CV scores are useful comparisons, but the difference between them is not a precise measurement of the effect of scaffold splitting.

In [ ]:
def compare_models(train_idx, groups):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    train_groups = groups.iloc[train_idx]
    cv = GroupKFold(n_splits=5)
    rows = []

    for name, model in models.items():
        print("Running:", name)
        scores = cross_validate(
            model, X_train, y_train,
            groups=train_groups, cv=cv, scoring=scoring,
            return_train_score=True, n_jobs=-1
        )
        rows.append({
            "Model": name,
            "Train RMSE": -scores["train_rmse"].mean(),
            "CV RMSE": -scores["test_rmse"].mean(),
            "CV RMSE SD": scores["test_rmse"].std(),
            "CV MAE": -scores["test_mae"].mean(),
            "CV R2": scores["test_r2"].mean()
        })

    return pd.DataFrame(rows).sort_values("CV RMSE").reset_index(drop=True)

random_results = compare_models(random_train_idx, connectivity)
print("\nConnectivity-group CV:")
display(random_results.round(3))

In [ ]:
scaffold_results = compare_models(scaffold_train_idx, combined_groups)
print("\nScaffold-group CV:")
display(scaffold_results.round(3))

## 5. Compare the models visually

First, we compare model accuracy across the two validation settings. Then we compare training and validation error on the scaffold setting to see which models may be overfitting.

In [ ]:
# Compare CV RMSE for all models
plot_data = random_results[["Model", "CV RMSE"]].rename(columns={"CV RMSE": "Connectivity CV"})
plot_data = plot_data.merge(
    scaffold_results[["Model", "CV RMSE"]].rename(columns={"CV RMSE": "Scaffold CV"}),
    on="Model"
).sort_values("Scaffold CV")

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = np.arange(len(plot_data))
ax.barh(y_pos - 0.2, plot_data["Connectivity CV"], height=0.4, label="Connectivity CV")
ax.barh(y_pos + 0.2, plot_data["Scaffold CV"], height=0.4, label="Scaffold CV")
ax.set_yticks(y_pos, plot_data["Model"])
ax.invert_yaxis()
ax.set_xlabel("CV RMSE (LogS) — lower is better")
ax.set_title("Model comparison")
ax.legend()
plt.tight_layout()

figures_dir = PROJECT_ROOT / "figures" / "modeling"
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Check whether models fit the training data much better than validation data
plot_data = scaffold_results.sort_values("CV RMSE")
y_pos = np.arange(len(plot_data))
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(y_pos - 0.2, plot_data["Train RMSE"], height=0.4, label="Training RMSE")
ax.barh(y_pos + 0.2, plot_data["CV RMSE"], height=0.4, label="Validation RMSE")
ax.set_yticks(y_pos, plot_data["Model"])
ax.invert_yaxis()
ax.set_xlabel("RMSE (LogS)")
ax.set_title("Training vs validation error (scaffold CV)")
ax.legend()
plt.tight_layout()
fig.savefig(figures_dir / "overfitting_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Save results and note the findings

We save the results, excluded structures, and exact train/test assignments so Notebook 04 can reuse them. The **test sets have not been evaluated** in this notebook.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

random_results.to_csv(results_dir / "model_comparison_random_cv.csv", index=False)
scaffold_results.to_csv(results_dir / "model_comparison_scaffold_cv.csv", index=False)
excluded.to_csv(results_dir / "excluded_invalid_structures.csv", index=False)

split_info = df_model[["ID", "InChIKey"]].copy()
split_info["row_index"] = np.arange(len(df_model))
split_info["connectivity_group"] = connectivity
split_info["scaffold_group"] = scaffold
split_info["combined_group"] = combined_groups
split_info["connectivity_split"] = "train"
split_info.loc[random_test_idx, "connectivity_split"] = "test"
split_info["scaffold_split"] = "train"
split_info.loc[scaffold_test_idx, "scaffold_split"] = "test"
split_info.to_csv(results_dir / "evaluation_splits.csv", index=False)

print("Best connectivity CV model:", random_results.iloc[0]["Model"])
print("Best scaffold CV model:", scaffold_results.iloc[0]["Model"])
print("Saved comparison tables, split assignments, and figures.")